# Pothole Detection using Smartphone Sensor Data

This notebook implements a machine learning pipeline for detecting potholes using accelerometer and gyroscope data. It is designed to be generic and adaptable to multiple datasets, as long as they include accelerometer and gyroscope signals.

In [ ]:

# =========================================
# 1. Setup & Imports
# =========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import joblib

# Optional: for GPS visualisation
import folium

# Set visualisation style
sns.set(style="whitegrid")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
import scipy.io
import numpy as np

mat_folder = "/content/drive/MyDrive/datasets/preprocessed_mat"
csv_output_folder = "/content/drive/MyDrive/datasets/converted_csv"

os.makedirs(csv_output_folder, exist_ok=True)

def convert_mat_to_csv(mat_file, output_folder):
    # Load .mat file
    mat = scipy.io.loadmat(mat_file)

    print(f"\nContents of {mat_file}:")
    print(mat.keys())

    # Use .get() with defaults to avoid KeyErrors
    df = pd.DataFrame({
        "timestamp": np.arange(len(mat.get("xlacc", []))).flatten(),  # no time field, so use index
        "acc_x": mat.get("xlacc", np.zeros(0)).flatten(),
        "acc_y": mat.get("ylacc", np.zeros(0)).flatten(),
        "acc_z": mat.get("zlacc", np.zeros(0)).flatten(),
        "gyro_x": mat.get("gyro_x", np.zeros(len(mat.get("xlacc", [])))).flatten(),
        "gyro_y": mat.get("gyro_y", np.zeros(len(mat.get("xlacc", [])))).flatten(),
        "gyro_z": mat.get("gyro_z", np.zeros(len(mat.get("xlacc", [])))).flatten(),
        "latitude": mat.get("lat", np.full(len(mat.get("xlacc", [])), np.nan)).flatten(),
        "longitude": mat.get("lon", np.full(len(mat.get("xlacc", [])), np.nan)).flatten(),
        "label": mat.get("label", ["unknown"] * len(mat.get("xlacc", []))).flatten()
    })

    # Save as CSV
    base_name = os.path.basename(mat_file).replace(".mat", ".csv")
    csv_path = os.path.join(output_folder, base_name)
    df.to_csv(csv_path, index=False)
    print(f"Saved {csv_path} with {len(df)} rows")

# Loop through all MAT files in the folder
for file in os.listdir(mat_folder):
    if file.endswith(".mat"):
        convert_mat_to_csv(os.path.join(mat_folder, file), csv_output_folder)



Contents of /content/drive/MyDrive/datasets/preprocessed_mat/Aligator cracking8_22-Feb-2021.mat:
dict_keys(['__header__', '__version__', '__globals__', 'label', 'xlacc', 'ylacc', 'zlacc'])
Saved /content/drive/MyDrive/datasets/converted_csv/Aligator cracking8_22-Feb-2021.csv with 218 rows

Contents of /content/drive/MyDrive/datasets/preprocessed_mat/Bump2_15-Feb-2021.mat:
dict_keys(['__header__', '__version__', '__globals__', 'label', 'xlacc', 'ylacc', 'zlacc'])
Saved /content/drive/MyDrive/datasets/converted_csv/Bump2_15-Feb-2021.csv with 179 rows

Contents of /content/drive/MyDrive/datasets/preprocessed_mat/Aligator cracking48_15-Feb-2021.mat:
dict_keys(['__header__', '__version__', '__globals__', 'label', 'xlacc', 'ylacc', 'zlacc'])
Saved /content/drive/MyDrive/datasets/converted_csv/Aligator cracking48_15-Feb-2021.csv with 215 rows

Contents of /content/drive/MyDrive/datasets/preprocessed_mat/Aligator cracking42_15-Feb-2021.mat:
dict_keys(['__header__', '__version__', '__globals__

In [ ]:
# =========================================
# 2. Load Dataset (from Drive)
# =========================================
import glob

# Path to converted CSVs (update if needed)
csv_folder = "/content/drive/MyDrive/datasets/converted_csv"

# Load and merge all CSVs
all_files = glob.glob(csv_folder + "/*.csv")
df_list = [pd.read_csv(f) for f in all_files]
print(df_list)
print(csv_folder)
data = pd.concat(df_list, ignore_index=True)

print("Data preview:")
display(data.head())
print("\nColumns:", data.columns)
if 'label' in data.columns:
    print("\nClass distribution:")
    print(data['label'].value_counts())


[     timestamp     acc_x     acc_y     acc_z  gyro_x  gyro_y  gyro_z  \
0            0 -0.176417 -0.620549 -0.085982     0.0     0.0     0.0   
1            1 -0.176417 -0.620549 -0.085982     0.0     0.0     0.0   
2            2  0.003149 -0.697168 -0.116622     0.0     0.0     0.0   
3            3  0.213338 -0.869844  0.051533     0.0     0.0     0.0   
4            4  0.213338 -0.869844  0.051533     0.0     0.0     0.0   
..         ...       ...       ...       ...     ...     ...     ...   
213        213 -2.639808 -1.353613 -1.038743     0.0     0.0     0.0   
214        214 -2.639808 -1.353613 -1.038743     0.0     0.0     0.0   
215        215 -1.754985 -0.198402 -0.140470     0.0     0.0     0.0   
216        216  0.229183  1.960346  0.815953     0.0     0.0     0.0   
217        217  0.949472  1.584080  0.535449     0.0     0.0     0.0   

     latitude  longitude  label  
0         NaN        NaN      1  
1         NaN        NaN      1  
2         NaN        NaN      1 

,timestamp,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,latitude,longitude,label
0,0,-0.176417,-0.620549,-0.085982,0.0,0.0,0.0,NaN,NaN,1
1,1,-0.176417,-0.620549,-0.085982,0.0,0.0,0.0,NaN,NaN,1
2,2,0.003149,-0.697168,-0.116622,0.0,0.0,0.0,NaN,NaN,1
3,3,0.213338,-0.869844,0.051533,0.0,0.0,0.0,NaN,NaN,1
4,4,0.213338,-0.869844,0.051533,0.0,0.0,0.0,NaN,NaN,1



Columns: Index(['timestamp', 'acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z',
       'latitude', 'longitude', 'label'],
      dtype='object')

Class distribution:
label
1    170154
0     43983
Name: count, dtype: int64


In [ ]:
print("Columns in dataframe:", data.columns.tolist())
print("Non-null counts:")
print(data.count())


Columns in dataframe: ['timestamp', 'acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'latitude', 'longitude', 'label']
Non-null counts:
timestamp    214137
acc_x        214137
acc_y        214137
acc_z        214137
gyro_x       214137
gyro_y       214137
gyro_z       214137
latitude          0
longitude         0
label        214137
dtype: int64


In [ ]:
import glob

csv_folder = "/content/drive/MyDrive/datasets/converted_csv"
all_files = glob.glob(csv_folder + "/*.csv")

print("Found CSV files:", len(all_files))
print(all_files[:5])  # preview first 5 filenames

df_list = []
for f in all_files:
    df = pd.read_csv(f)
    print(f, df.shape)
    df_list.append(df)

data = pd.concat(df_list, ignore_index=True)
print("Final merged data shape:", data.shape)


Found CSV files: 920
['/content/drive/MyDrive/datasets/converted_csv/Aligator cracking8_22-Feb-2021.csv', '/content/drive/MyDrive/datasets/converted_csv/Bump2_15-Feb-2021.csv', '/content/drive/MyDrive/datasets/converted_csv/Aligator cracking48_15-Feb-2021.csv', '/content/drive/MyDrive/datasets/converted_csv/Aligator cracking42_15-Feb-2021.csv', '/content/drive/MyDrive/datasets/converted_csv/Bump11_22-Feb-2021.csv']
/content/drive/MyDrive/datasets/converted_csv/Aligator cracking8_22-Feb-2021.csv (218, 10)
/content/drive/MyDrive/datasets/converted_csv/Bump2_15-Feb-2021.csv (179, 10)
/content/drive/MyDrive/datasets/converted_csv/Aligator cracking48_15-Feb-2021.csv (215, 10)
/content/drive/MyDrive/datasets/converted_csv/Aligator cracking42_15-Feb-2021.csv (266, 10)
/content/drive/MyDrive/datasets/converted_csv/Bump11_22-Feb-2021.csv (178, 10)
/content/drive/MyDrive/datasets/converted_csv/Aligator cracking51_15-Feb-2021.csv (279, 10)
/content/drive/MyDrive/datasets/converted_csv/Aligator cr

In [ ]:
print("Columns in merged data:", data.columns.tolist())
print(data.head())


Columns in merged data: ['timestamp', 'acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z', 'latitude', 'longitude', 'label']
   timestamp     acc_x     acc_y     acc_z  gyro_x  gyro_y  gyro_z  latitude  \
0          0 -0.176417 -0.620549 -0.085982     0.0     0.0     0.0       NaN   
1          1 -0.176417 -0.620549 -0.085982     0.0     0.0     0.0       NaN   
2          2  0.003149 -0.697168 -0.116622     0.0     0.0     0.0       NaN   
3          3  0.213338 -0.869844  0.051533     0.0     0.0     0.0       NaN   
4          4  0.213338 -0.869844  0.051533     0.0     0.0     0.0       NaN   

   longitude  label  
0        NaN      1  
1        NaN      1  
2        NaN      1  
3        NaN      1  
4        NaN      1  


In [ ]:
df_list = []
for f in all_files:
    df = pd.read_csv(f)
    if df.shape[0] > 0:   # only keep non-empty
        df_list.append(df)

if df_list:
    data = pd.concat(df_list, ignore_index=True)
else:
    raise ValueError("No non-empty CSVs found in folder!")


In [ ]:
# =========================================
# 3. Preprocessing
# =========================================
# Drop missing values
data = data.dropna()

# Normalise accelerometer & gyroscope values
scaler = StandardScaler()
sensor_cols = ['acc_x','acc_y','acc_z','gyro_x','gyro_y','gyro_z']
data[sensor_cols] = scaler.fit_transform(data[sensor_cols])

# Optional: filter signal to reduce noise (example low-pass filter)
fs = 50  # Hz, adjust to dataset sampling rate
cutoff = 5  # Hz
b, a = signal.butter(3, cutoff/(0.5*fs), btype='low')
for col in sensor_cols:
    data[col] = signal.filtfilt(b, a, data[col])


ValueError: Found array with 0 sample(s) (shape=(0, 6)) while a minimum of 1 is required by StandardScaler.

In [ ]:
# =========================================
# 4. Feature Extraction (windowing)
# =========================================
def extract_features(df, window_size=100, overlap=0.5):
    features, labels = [], []
    step = int(window_size * (1 - overlap))
    for start in range(0, len(df) - window_size, step):
        end = start + window_size
        window = df.iloc[start:end]
        if 'label' not in window.columns:   # safety check
            continue
        label = window['label'].mode()[0]  # most frequent label in window
        feat = []
        for col in sensor_cols:
            signal_data = window[col].values
            feat.append(np.mean(signal_data))
            feat.append(np.std(signal_data))
            feat.append(np.max(signal_data))
            feat.append(np.min(signal_data))
            feat.append(np.sqrt(np.mean(signal_data**2)))  # RMS
            # Frequency domain feature (dominant frequency)
            freqs = np.fft.rfftfreq(len(signal_data), 1/fs)
            fft_magnitude = np.abs(np.fft.rfft(signal_data))
            dom_freq = freqs[np.argmax(fft_magnitude)]
            feat.append(dom_freq)
        features.append(feat)
        labels.append(label)
    return np.array(features), np.array(labels)

X, y = extract_features(data, window_size=128, overlap=0.5)
print("Feature shape:", X.shape, "Label shape:", y.shape)

In [ ]:
# =========================================
# 5. Train/Test Split
# =========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
# =========================================
# 6. Model Training
# =========================================
# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# SVM
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train, y_train)

In [ ]:
# =========================================
# 7. Evaluation
# =========================================
models = {'Decision Tree': dt, 'Random Forest': rf, 'SVM': svm}

for name, model in models.items():
    y_pred = model.predict(X_test)
    print(f"\n{name} Results:")
    print(classification_report(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=np.unique(y),
                yticklabels=np.unique(y))
    plt.title(f"{name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()


In [ ]:
# =========================================
# 9. Save Best Model
# =========================================
best_model = rf  # choose based on evaluation
joblib.dump(best_model, "/content/drive/MyDrive/datasets/pothole_detector.pkl")
print("Model saved to Drive as pothole_detector.pkl")

In [ ]:
# =========================================
# Extra: Visual sanity check of signals
# =========================================
sample = data.head(500)  # first 500 rows as a slice

plt.figure(figsize=(12,6))
plt.plot(sample['acc_x'], label='acc_x')
plt.plot(sample['acc_y'], label='acc_y')
plt.plot(sample['acc_z'], label='acc_z')
plt.title("Raw accelerometer signals (first 500 samples)")
plt.xlabel("Sample index")
plt.ylabel("Normalised value")
plt.legend()
plt.show()

# If labels are available
if 'label' in sample.columns:
    plt.figure(figsize=(12,3))
    plt.plot(sample['label'].astype(str).values, 'r.')
    plt.title("Labels (first 500 samples)")
    plt.show()
